In [ ]:
import pandas as pd
import numpy as np
import json
import time
import re
import os
import math
import warnings
from datetime import datetime
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings('ignore')

# =============================================================================
# GLOBAL CONFIGURATION
# =============================================================================

CONFIG = {
    # ---- API Keys ----
    "GRAPH_API_KEY": "7c05cbf91f535db4bfb56b7d9da40208",  # <-- PASTE YOUR GRAPH API KEY HERE

    # ---- Directories ----
    "UMA_OUTPUT_DIR": "./prediction_market_data",
    "POLYMARKET_OUTPUT_DIR": "./polymarket_data",
    "SRS_OUTPUT_DIR": "./polymarket_srs_predictions",
    "RESULTS_DIR": "./results",

    # ---- UMA Collection ----
    "MAX_UMA_PRICE_REQUESTS": 100000,
    "UMA_BATCH_SIZE": 100,
    "UMA_REQUEST_DELAY": 0.5,
    "UMA_CHECKPOINT_EVERY": 1000,
    "NUM_FEWSHOT_LOW": 9,
    "NUM_FEWSHOT_HIGH": 1,
    "RANDOM_SEED": 42,

    # ---- Polymarket Collection ----
    "GAMMA_API_BASE": "https://gamma-api.polymarket.com",
    "CLOB_API_BASE": "https://clob.polymarket.com",
    "PM_MAX_MARKETS": 1000,
    "PM_OFFSET_START": 8000,
    "PM_REQUEST_DELAY": 0.2,
    "PM_MIN_VOLUME": 5000,
    "PM_MIN_YEAR": 2022,

    # ---- Model Settings ----
    "PRIMARY_MODEL": "Qwen/Qwen3-8B",
    "SECONDARY_MODEL": "meta-llama/Meta-Llama-3-8B-Instruct",  # For robustness
    "QUANTIZATION": "4bit",
    "MAX_NEW_TOKENS": 300,
    "TEMPERATURE_PRIMARY": 0.0,
    "TEMPERATURE_SENSITIVITY": 0.6,  # For sensitivity check

    # ---- Processing ----
    "CHECKPOINT_EVERY": 50,

    # ---- Portfolio ----
    "SRS_THRESHOLD": 0.10,
    "GAMMA": 1,
    "MIN_VIABLE_VOLUME": 5000,
}

# =============================================================================
# UMA API SETUP
# =============================================================================

UMA_SUBGRAPH_ID = "5YVXjj28Lv4eLhHg54R1QWVHNn8VAjZnT3vJgEtuyWmY"

def get_uma_url():
    key = CONFIG["GRAPH_API_KEY"]
    if not key:
        return None
    return f"https://gateway.thegraph.com/api/{key}/subgraphs/id/{UMA_SUBGRAPH_ID}"

def query_subgraph(url, query):
    import requests
    if not url:
        return {"error": "No URL (missing API key?)"}
    try:
        response = requests.post(url, json={"query": query}, timeout=60)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": str(e)}

# =============================================================================
# STAGE 1: UMA DATA COLLECTION
# =============================================================================

def stage_1_collect_uma():
    """Collect UMA dispute data from The Graph."""
    import requests

    print("\n" + "=" * 70)
    print("STAGE 1: UMA DATA COLLECTION")
    print("=" * 70)

    output_dir = CONFIG["UMA_OUTPUT_DIR"]
    os.makedirs(output_dir, exist_ok=True)

    if not CONFIG["GRAPH_API_KEY"]:
        print("ERROR: No Graph API key set. Set CONFIG['GRAPH_API_KEY'].")
        return None

    # Test connection
    url = get_uma_url()
    test = query_subgraph(url, "{ priceRequests(first: 1) { id } }")
    if "error" in test or "errors" in test:
        print(f"Connection failed: {test}")
        return None
    print("UMA connection successful.")

    # Check for existing data
    raw_path = os.path.join(output_dir, "uma_final.json")
    if os.path.exists(raw_path):
        print(f"Loading existing raw data: {raw_path}")
        with open(raw_path, 'r') as f:
            all_requests = json.load(f)
        print(f"Loaded {len(all_requests)} requests.")
    else:
        # Fetch all price requests
        all_requests = []
        skip = 0
        batch_size = CONFIG["UMA_BATCH_SIZE"]
        max_requests = CONFIG["MAX_UMA_PRICE_REQUESTS"]
        empty_count = 0

        while len(all_requests) < max_requests:
            query = f"""
            {{
              priceRequests(first: {batch_size}, skip: {skip}, orderBy: time, orderDirection: desc) {{
                id
                isResolved
                price
                time
                resolutionTimestamp
                identifier {{ id }}
                ancillaryData
                latestRound {{
                  id
                  roundId
                  totalVotesRevealed
                  groups {{ price totalVoteAmount }}
                }}
                rounds {{ id roundId totalVotesRevealed }}
              }}
            }}
            """
            result = query_subgraph(url, query)
            batch = result.get("data", {}).get("priceRequests", [])

            if not batch:
                empty_count += 1
                if empty_count >= 3:
                    break
                time.sleep(2)
                continue

            empty_count = 0
            all_requests.extend(batch)
            print(f"  Offset {skip}: Got {len(batch)}. Total: {len(all_requests)}")
            skip += len(batch)
            time.sleep(CONFIG["UMA_REQUEST_DELAY"])

            if len(all_requests) % CONFIG["UMA_CHECKPOINT_EVERY"] < batch_size:
                with open(os.path.join(output_dir, "uma_checkpoint.json"), 'w') as f:
                    json.dump(all_requests, f, default=str)

        with open(raw_path, 'w') as f:
            json.dump(all_requests, f, default=str)
        print(f"Saved {len(all_requests)} raw requests.")

    # Process into DataFrame
    def decode_hex(hex_data):
        if not hex_data:
            return ""
        try:
            if hex_data.startswith('0x'):
                hex_data = hex_data[2:]
            return bytes.fromhex(hex_data).decode('utf-8', errors='ignore')
        except:
            return ""

    def decode_question(request_id):
        try:
            if '0x' in str(request_id):
                hex_part = str(request_id).split('0x', 1)[1]
                return bytes.fromhex(hex_part).decode('utf-8', errors='ignore')
        except:
            pass
        return ""

    def bernoulli_variance(groups):
        if not groups or len(groups) < 2:
            return 0.0
        try:
            amounts = [float(g.get('totalVoteAmount', 0)) for g in groups]
            total = sum(amounts)
            if total == 0:
                return 0.0
            p = amounts[0] / total
            return p * (1 - p)
        except:
            return 0.0

    rows = []
    for r in all_requests:
        latest = r.get('latestRound') or {}
        groups = latest.get('groups') or []
        ident = r.get('identifier') or {}
        request_id = r.get('id', '')

        rows.append({
            'uma_request_id': request_id,
            'uma_identifier': ident.get('id', ''),
            'uma_ancillary_data_decoded': decode_hex(r.get('ancillaryData', '')),
            'question_decoded': decode_question(request_id),
            'uma_timestamp': r.get('time', ''),
            'uma_resolution_timestamp': r.get('resolutionTimestamp', ''),
            'uma_is_resolved': r.get('isResolved', False),
            'uma_price': r.get('price', ''),
            'uma_total_votes_revealed': latest.get('totalVotesRevealed', 0),
            'uma_num_vote_groups': len(groups),
            'uma_num_rounds': len(r.get('rounds') or []),
            'uma_bernoulli_variance': bernoulli_variance(groups),
            'uma_vote_groups_raw': json.dumps(groups),
        })

    df = pd.DataFrame(rows)
    has_question = df['question_decoded'].str.contains('title:', case=False, na=False)
    df = df[has_question].copy()

    # Add variance bins
    df['variance_bin'] = pd.cut(
        df['uma_bernoulli_variance'],
        bins=[-0.001, 0.001, 0.05, 0.15, 0.26],
        labels=['zero', 'low', 'medium', 'high']
    )

    save_path = os.path.join(output_dir, "01_uma_all_markets.csv")
    df.to_csv(save_path, index=False)
    print(f"\nProcessed {len(df)} UMA markets with decoded questions.")
    print(f"Saved to {save_path}")

    return df


# =============================================================================
# STAGE 2: UMA CLEANUP & FEW-SHOT SPLIT
# =============================================================================

def stage_2_uma_cleanup(uma_df=None):
    """Split UMA data into few-shot examples and analysis set."""
    print("\n" + "=" * 70)
    print("STAGE 2: UMA CLEANUP & FEW-SHOT SPLIT")
    print("=" * 70)

    output_dir = CONFIG["UMA_OUTPUT_DIR"]

    if uma_df is None:
        path = os.path.join(output_dir, "01_uma_all_markets.csv")
        uma_df = pd.read_csv(path)
        print(f"Loaded {len(uma_df)} markets from {path}")

    # The 10 curated few-shot examples (5 zero + 2 low + 2 medium + 1 high)
    few_shot_titles = [
        "MLB: New York Mets vs. Atlanta Braves 2023-04-29",
        "Evansville vs. UIC",
        "Will Aston Villa vs. Paris Saint Germain end in a draw?",
        "Hamas leadership out of Qatar before Trump in office?",
        "Ducks vs. Red Wings",
        "Will the highest temperature in London be 53°F or below on April 2?",
        "U.S. military action against Yemen in 2024?",
        "Will Trump launch a coin before the election?",
        "Will Trump lower tariffs on China in April?",
        "Farcaster unique users less than 15k on Feb 5?"
    ]

    def normalize_title(text):
        if not isinstance(text, str):
            return ""
        match = re.search(r"title:\s*(.*?)(?:,|description:|$)", text, re.IGNORECASE | re.DOTALL)
        raw_title = match.group(1) if match else text
        return re.sub(r'[^a-zA-Z0-9]', '', raw_title).lower()

    uma_df['normalized_title'] = uma_df['question_decoded'].apply(normalize_title)
    target_hashes = set(re.sub(r'[^a-zA-Z0-9]', '', t).lower() for t in few_shot_titles)

    is_few_shot = uma_df['normalized_title'].isin(target_hashes)
    few_shot_df = uma_df[is_few_shot].copy()
    analysis_df = uma_df[~is_few_shot].copy()

    print(f"Few-Shot Set: {len(few_shot_df)} markets")
    print(f"Analysis Set: {len(analysis_df)} markets")

    if len(few_shot_df) < 10:
        print(f"WARNING: Only found {len(few_shot_df)} out of 10 examples.")
        found = set(few_shot_df['normalized_title'])
        for t in few_shot_titles:
            norm_t = re.sub(r'[^a-zA-Z0-9]', '', t).lower()
            if norm_t not in found:
                print(f"  Missing: {t}")
    else:
        print("All 10 few-shot examples identified.")

    few_shot_df.drop(columns=['normalized_title'], inplace=True, errors='ignore')
    analysis_df.drop(columns=['normalized_title'], inplace=True, errors='ignore')

    few_shot_df.to_csv(os.path.join(output_dir, "02_fewshot_examples_v11.csv"), index=False)
    analysis_df.to_csv(os.path.join(output_dir, "03_analysis_set.csv"), index=False)

    print(f"Saved to {output_dir}/")
    return few_shot_df, analysis_df


# =============================================================================
# SHARED: MODEL LOADING & SRS PREDICTION
# =============================================================================

FEW_SHOT_EXAMPLES_TEXT = """
## REAL EXAMPLES FROM UMA DISPUTE DATA
## Distribution: 5 Zero + 2 Low + 2 Medium + 1 High

Example 1 (Variance: 0.00): "MLB: New York Mets vs. Atlanta Braves 2023-04-29"
Analysis: Sports game outcome with official MLB results. Binary win/loss with no ambiguity.
PREDICTED VARIANCE: 0.00

Example 2 (Variance: 0.00): "Evansville vs. UIC"
Analysis: Sports matchup with clear official outcome. No interpretation needed.
PREDICTED VARIANCE: 0.00

Example 3 (Variance: 0.00): "Will Aston Villa vs. Paris Saint Germain end in a draw?"
Analysis: Soccer match outcome - draw is clearly defined. Official result determines resolution.
PREDICTED VARIANCE: 0.00

Example 4 (Variance: 0.00): "Hamas leadership out of Qatar before Trump in office?"
Analysis: Verifiable event with specific deadline (inauguration date). Binary yes/no.
PREDICTED VARIANCE: 0.00

Example 5 (Variance: 0.00): "Ducks vs. Red Wings"
Analysis: NHL game outcome. Official score determines winner with no ambiguity.
PREDICTED VARIANCE: 0.00

Example 6 (Variance: 0.02): "Will the highest temperature in London be 53°F or below on April 2?"
Analysis: Specific temperature threshold, specific location, specific date. Minor ambiguity about which weather source to use.
PREDICTED VARIANCE: 0.02

Example 7 (Variance: 0.03): "U.S. military action against Yemen in 2024?"
Analysis: Observable military event, but "military action" could have varying interpretations (strikes vs advisors vs full engagement).
PREDICTED VARIANCE: 0.03

Example 8 (Variance: 0.08): "Will Trump launch a coin before the election?"
Analysis: "Launch a coin" is ambiguous - official crypto token? Meme coin? Physical commemorative coin? "Before election" needs specific date.
PREDICTED VARIANCE: 0.08

Example 9 (Variance: 0.10): "Will Trump lower tariffs on China in April?"
Analysis: "Lower tariffs" is vague - any reduction? Specific categories? "April" of which year? Multiple interpretations possible.
PREDICTED VARIANCE: 0.10

Example 10 (Variance: 0.21): "Farcaster unique users less than 15k on Feb 5?"
Analysis: "Unique users" definition unclear - daily active? Total registered? Which timezone for Feb 5? No official source specified.
PREDICTED VARIANCE: 0.21
"""

def get_system_prompt():
    return f"""You predict dispute variance for prediction market contracts.

SCALE:
- 0.00 = Perfect clarity, all resolvers would agree (e.g., sports scores)
- 0.25 = Maximum ambiguity, resolvers split 50/50

KEY INSIGHT: Most prediction market contracts (~87%) have variance at or near 0.00 because they are well-specified.

FACTORS THAT KEEP VARIANCE LOW (0.00-0.03):
- Sports outcomes (official scores)
- Election results (official counts)
- Specific price thresholds with named exchanges
- Exact dates and times
- Named official data sources

FACTORS THAT INCREASE VARIANCE (0.05+):
- Vague qualifiers ("significant", "major", "meaningful")
- No resolution source specified
- Subjective terms ("launch", "action", "improve")
- Complex multi-part conditions
- Edge cases not addressed

{FEW_SHOT_EXAMPLES_TEXT}

TASK: Analyze the contract below and predict its dispute variance.
Output: Brief analysis (1-2 sentences), then "PREDICTED VARIANCE: X.XX" on the final line."""


def clear_gpu_memory():
    """Clear GPU memory."""
    import torch
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


def load_model(model_name=None):
    """Load a quantized LLM."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if model_name is None:
        model_name = CONFIG["PRIMARY_MODEL"]

    print(f"Loading model: {model_name}")

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name, trust_remote_code=True, padding_side="left"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )

    print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    return model, tokenizer


def predict_srs(model, tokenizer, contract_text, temperature=0.0,
                model_name=None, is_llama=False):
    """Predict Semantic Risk Score for a contract."""
    import torch

    contract_truncated = contract_text[:2000]

    # Adjust prompt for Llama vs Qwen
    if is_llama:
        user_prompt = f"""Predict dispute variance for this contract:

CONTRACT: {contract_truncated}

Remember: Most contracts have variance near 0.00 (like sports outcomes). Only predict higher if there's genuine ambiguity in the resolution criteria.

Brief analysis, then final line: "PREDICTED VARIANCE: X.XX"
"""
    else:
        user_prompt = f"""Analyze this prediction market contract and predict its dispute variance:

CONTRACT: {contract_truncated}

Remember: Most contracts have variance near 0.00 (like sports outcomes). Only predict higher if there's genuine ambiguity in the resolution criteria.

Brief analysis, then final line: "PREDICTED VARIANCE: X.XX"
/no_think"""

    messages = [
        {"role": "system", "content": get_system_prompt()},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        if temperature > 0:
            outputs = model.generate(
                **inputs,
                max_new_tokens=CONFIG["MAX_NEW_TOKENS"],
                temperature=temperature,
                do_sample=True, top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
            )
        else:
            outputs = model.generate(
                **inputs,
                max_new_tokens=CONFIG["MAX_NEW_TOKENS"],
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    variance = extract_variance(response)

    return {"response": response[:500], "predicted_srs": variance}


def extract_variance(response):
    """Extract variance from model response."""
    patterns = [
        r'PREDICTED\s*VARIANCE:\s*(0\.\d+)',
        r'PREDICTED\s*VARIANCE:\s*(0)',
        r'[Vv]ariance:\s*(0\.\d+)',
        r'\*\*(0\.\d+)\*\*',
    ]
    for pattern in patterns:
        match = re.search(pattern, response)
        if match:
            try:
                value = float(match.group(1))
                return max(0.0, min(0.25, value))
            except:
                continue

    decimals = re.findall(r'(0\.\d+)', response[-100:])
    for d in reversed(decimals):
        try:
            value = float(d)
            if 0.0 <= value <= 0.30:
                return min(0.25, value)
        except:
            continue
    return None


# =============================================================================
# STAGE 3: SRS VALIDATION ON UMA
# =============================================================================

def stage_3_validate_uma(analysis_df=None):
    """Validate SRS predictions against UMA ground truth."""
    from tqdm import tqdm

    print("\n" + "=" * 70)
    print("STAGE 3: SRS VALIDATION ON UMA (Primary Model)")
    print("=" * 70)

    output_dir = CONFIG["UMA_OUTPUT_DIR"]
    os.makedirs(output_dir, exist_ok=True)

    if analysis_df is None:
        path = os.path.join(output_dir, "03_analysis_set.csv")
        analysis_df = pd.read_csv(path)
        print(f"Loaded {len(analysis_df)} markets from {path}")

    # Load model
    clear_gpu_memory()
    model, tokenizer = load_model(CONFIG["PRIMARY_MODEL"])

    results = []
    for idx, row in tqdm(analysis_df.iterrows(), total=len(analysis_df), desc="Validating SRS"):
        market_text = row.get('question_decoded', '')
        if not market_text or len(market_text) < 20:
            results.append({
                'uma_request_id': row.get('uma_request_id', f'market_{idx}'),
                'predicted_srs': None, 'response': 'SKIPPED'
            })
            continue

        result = predict_srs(model, tokenizer, market_text,
                             temperature=CONFIG["TEMPERATURE_PRIMARY"])
        results.append({
            'uma_request_id': row.get('uma_request_id', f'market_{idx}'),
            'predicted_srs': result['predicted_srs'],
            'response': result['response'],
        })

        if len(results) % CONFIG["CHECKPOINT_EVERY"] == 0:
            pd.DataFrame(results).to_csv(
                os.path.join(output_dir, f"validation_checkpoint_{len(results)}.csv"),
                index=False
            )
            print(f"  Checkpoint: {len(results)} predictions")

    results_df = pd.DataFrame(results)

    # Merge with ground truth
    merged = analysis_df.merge(
        results_df[['uma_request_id', 'predicted_srs']],
        on='uma_request_id', how='left'
    )

    # Validation metrics
    valid = merged.dropna(subset=['predicted_srs', 'uma_bernoulli_variance'])
    if len(valid) >= 3:
        from scipy.stats import spearmanr, pearsonr

        predicted = valid['predicted_srs']
        actual = valid['uma_bernoulli_variance']

        sp_corr, sp_p = spearmanr(predicted, actual)
        pe_corr, pe_p = pearsonr(predicted, actual)
        mae = (predicted - actual).abs().mean()

        print(f"\n{'='*60}")
        print("VALIDATION RESULTS")
        print(f"{'='*60}")
        print(f"N = {len(valid)}")
        print(f"Spearman ρ = {sp_corr:.4f} (p = {sp_p:.6f})")
        print(f"Pearson r  = {pe_corr:.4f} (p = {pe_p:.6f})")
        print(f"MAE        = {mae:.4f}")

    save_path = os.path.join(output_dir, "validation_results.csv")
    merged.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

    # Cleanup GPU
    del model, tokenizer
    clear_gpu_memory()

    return merged


# =============================================================================
# STAGE 4: POLYMARKET DATA COLLECTION
# =============================================================================

def stage_4_collect_polymarket():
    """Collect Polymarket resolved market data."""
    import requests
    from tqdm import tqdm

    print("\n" + "=" * 70)
    print("STAGE 4: POLYMARKET DATA COLLECTION")
    print("=" * 70)

    output_dir = CONFIG["POLYMARKET_OUTPUT_DIR"]
    os.makedirs(output_dir, exist_ok=True)

    # Check for existing raw data
    raw_path = os.path.join(output_dir, "01_raw_markets_pilot.json")
    if os.path.exists(raw_path):
        print(f"Loading existing raw data: {raw_path}")
        with open(raw_path, 'r') as f:
            all_markets = json.load(f)
    else:
        # Fetch markets
        all_markets = []
        offset = CONFIG["PM_OFFSET_START"]
        limit = 100
        max_offset = offset + 50000

        while len(all_markets) < CONFIG['PM_MAX_MARKETS'] and offset < max_offset:
            try:
                url = f"{CONFIG['GAMMA_API_BASE']}/markets"
                params = {"limit": limit, "offset": offset, "closed": "true"}
                response = requests.get(url, params=params, timeout=30)
                response.raise_for_status()
                batch = response.json()
            except:
                batch = []

            if not batch:
                break

            valid_batch = []
            for m in batch:
                try:
                    created = m.get('createdAt') or m.get('created_at')
                    if created and int(created[:4]) >= CONFIG['PM_MIN_YEAR']:
                        valid_batch.append(m)
                except:
                    pass

            all_markets.extend(valid_batch)
            print(f"  Offset {offset}: Got {len(batch)}, kept {len(valid_batch)}. Total: {len(all_markets)}")
            offset += limit
            time.sleep(CONFIG["PM_REQUEST_DELAY"])
            if len(batch) < limit:
                break

        all_markets = all_markets[:CONFIG['PM_MAX_MARKETS']]
        with open(raw_path, 'w') as f:
            json.dump(all_markets, f)

    # Process markets
    def process_market(market):
        try:
            question = market.get('question', '')
            yes_token_id = None
            outcome_resolved = market.get('outcome')

            tokens = market.get('tokens', [])
            if isinstance(tokens, str):
                try: tokens = json.loads(tokens)
                except: tokens = []

            if tokens and isinstance(tokens, list):
                for token in tokens:
                    if not isinstance(token, dict): continue
                    outcome_label = str(token.get('outcome', '')).lower()
                    tid = token.get('token_id') or token.get('tokenId')
                    if not outcome_resolved and token.get('winner') is True:
                        outcome_resolved = token.get('outcome')
                    if outcome_label == 'yes':
                        yes_token_id = tid

            if not yes_token_id:
                clob_ids = market.get('clobTokenIds', [])
                outcomes = market.get('outcomes', [])
                if isinstance(clob_ids, str):
                    try: clob_ids = json.loads(clob_ids)
                    except: clob_ids = []
                if isinstance(outcomes, str):
                    try: outcomes = json.loads(outcomes)
                    except: outcomes = []
                if clob_ids and outcomes and len(clob_ids) == len(outcomes):
                    for i, out_val in enumerate(outcomes):
                        if str(out_val).lower() == 'yes':
                            yes_token_id = clob_ids[i]

            if not yes_token_id:
                return None

            return {
                'condition_id': market.get('conditionId'),
                'question': question,
                'description': market.get('description', ''),
                'yes_token_id': yes_token_id,
                'volume': float(market.get('volume', 0)),
                'created_at': market.get('createdAt'),
                'resolved_at': market.get('endDate'),
                'outcome_resolved': outcome_resolved
            }
        except:
            return None

    processed = []
    for m in all_markets:
        p = process_market(m)
        if p:
            processed.append(p)

    df = pd.DataFrame(processed)
    print(f"Valid markets: {len(df)}")

    df = df[df['volume'] >= CONFIG['PM_MIN_VOLUME']]
    print(f"After volume filter (>${CONFIG['PM_MIN_VOLUME']}): {len(df)}")

    if df.empty:
        print("No markets passed filters.")
        return None

    # Fetch price histories
    def fetch_price_history_chunked(token_id, start_ts, end_ts):
        url = f"{CONFIG['CLOB_API_BASE']}/prices-history"
        CHUNK_SIZE = 30 * 24 * 60 * 60
        headers = {"User-Agent": "Mozilla/5.0"}
        all_history = []
        current_start = start_ts

        while current_start < end_ts:
            current_end = min(current_start + CHUNK_SIZE, end_ts)
            if current_end - current_start < 60:
                break
            params = {"market": token_id, "startTs": current_start,
                      "endTs": current_end, "fidelity": 60}
            try:
                resp = requests.get(url, params=params, headers=headers, timeout=10)
                if resp.status_code == 200:
                    all_history.extend(resp.json().get("history", []))
            except:
                pass
            current_start = current_end
            time.sleep(0.1)

        if not all_history:
            return pd.DataFrame()

        pdf = pd.DataFrame(all_history)
        if 't' in pdf.columns and 'p' in pdf.columns:
            pdf['timestamp'] = pd.to_datetime(pdf['t'], unit='s')
            pdf['price'] = pdf['p'].astype(float)
            pdf = pdf.sort_values('timestamp').drop_duplicates('timestamp')
            return pdf
        return pd.DataFrame()

    def calculate_metrics(price_df, outcome_res):
        metrics = {'price_count': len(price_df), 'final_price': None,
                   'volatility_24h': None, 'resolution_jump': None}
        if price_df.empty:
            return metrics
        metrics['final_price'] = price_df['price'].iloc[-1]
        try:
            price_df = price_df.copy()
            price_df['returns'] = price_df['price'].pct_change()
            if len(price_df) > 5:
                window = min(len(price_df), 24)
                metrics['volatility_24h'] = price_df.tail(window)['returns'].std()
            out_res = str(outcome_res).lower() if outcome_res else ""
            target = None
            if out_res == 'yes': target = 1.0
            elif out_res == 'no': target = 0.0
            if target is not None:
                metrics['resolution_jump'] = abs(target - metrics['final_price'])
        except:
            pass
        return metrics

    print("\nFetching price histories...")
    metrics_list = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            end_ts = int(pd.to_datetime(row['resolved_at']).timestamp())
            start_ts = int(pd.to_datetime(row['created_at']).timestamp())
        except:
            metrics_list.append({})
            continue
        price_df = fetch_price_history_chunked(row['yes_token_id'], start_ts, end_ts)
        m = calculate_metrics(price_df, row['outcome_resolved'])
        m['condition_id'] = row['condition_id']
        metrics_list.append(m)

    metrics_df = pd.DataFrame(metrics_list)
    final_df = df.merge(metrics_df, on='condition_id', how='left')

    save_path = os.path.join(output_dir, "05_analysis_ready.csv")
    final_df.to_csv(save_path, index=False)
    print(f"\nSaved {len(final_df)} markets to {save_path}")

    return final_df


# =============================================================================
# STAGE 5: POLYMARKET CLEANUP
# =============================================================================

def stage_5_clean_polymarket(pm_df=None):
    """Clean Polymarket data for SRS prediction."""
    print("\n" + "=" * 70)
    print("STAGE 5: POLYMARKET CLEANUP")
    print("=" * 70)

    output_dir = CONFIG["POLYMARKET_OUTPUT_DIR"]

    if pm_df is None:
        path = os.path.join(output_dir, "05_analysis_ready.csv")
        pm_df = pd.read_csv(path)

    print(f"Original count: {len(pm_df)}")
    df_clean = pm_df[pm_df['volatility_24h'].notna()].copy()

    if 'description' not in df_clean.columns:
        df_clean['description'] = ""
        print("Added placeholder 'description' column.")

    save_path = os.path.join(output_dir, "05_analysis_ready.csv")
    df_clean.to_csv(save_path, index=False)
    print(f"Ready for SRS prediction: {len(df_clean)} markets")

    return df_clean


# =============================================================================
# STAGE 6: SRS PREDICTION ON POLYMARKET
# =============================================================================

def stage_6_predict_polymarket(pm_df=None, model_name=None,
                                temperature=None, output_suffix=""):
    """Run SRS prediction on Polymarket contracts.

    This is reusable for both the primary model and the secondary model.
    """
    from tqdm import tqdm

    if model_name is None:
        model_name = CONFIG["PRIMARY_MODEL"]
    if temperature is None:
        temperature = CONFIG["TEMPERATURE_PRIMARY"]

    is_llama = "llama" in model_name.lower()
    label = f"({os.path.basename(model_name)}, temp={temperature})"

    print("\n" + "=" * 70)
    print(f"STAGE 6: SRS PREDICTION ON POLYMARKET {label}")
    print("=" * 70)

    pm_dir = CONFIG["POLYMARKET_OUTPUT_DIR"]
    srs_dir = CONFIG["SRS_OUTPUT_DIR"]
    os.makedirs(srs_dir, exist_ok=True)

    if pm_df is None:
        path = os.path.join(pm_dir, "05_analysis_ready.csv")
        pm_df = pd.read_csv(path)
        print(f"Loaded {len(pm_df)} markets")

    # Load model
    clear_gpu_memory()
    model, tokenizer = load_model(model_name)

    results = []
    for idx, row in tqdm(pm_df.iterrows(), total=len(pm_df), desc="Predicting SRS"):
        condition_id = row.get('condition_id', f'market_{idx}')
        question = row.get('question', '')
        description = row.get('description', '')
        contract_text = f"Question: {question}\n\nDescription: {description}"

        if len(contract_text.strip()) < 20:
            results.append({
                'condition_id': condition_id,
                'predicted_srs': None,
                'response': 'SKIPPED'
            })
            continue

        start_time = time.time()
        result = predict_srs(model, tokenizer, contract_text,
                             temperature=temperature, is_llama=is_llama)
        elapsed = time.time() - start_time

        results.append({
            'condition_id': condition_id,
            'predicted_srs': result['predicted_srs'],
            'response': result['response'],
            'elapsed_seconds': elapsed,
        })

        if len(results) % CONFIG["CHECKPOINT_EVERY"] == 0:
            pd.DataFrame(results).to_csv(
                os.path.join(srs_dir, f"checkpoint{output_suffix}_{len(results)}.csv"),
                index=False
            )

    results_df = pd.DataFrame(results)
    pred_path = os.path.join(srs_dir, f"srs_predictions{output_suffix}.csv")
    results_df.to_csv(pred_path, index=False)

    # Merge with original
    srs_col = f"predicted_srs{output_suffix}" if output_suffix else "predicted_srs"
    merge_df = results_df[['condition_id', 'predicted_srs']].rename(
        columns={'predicted_srs': srs_col}
    )
    final_df = pm_df.merge(merge_df, on='condition_id', how='left')

    final_path = os.path.join(srs_dir, f"analysis_with_srs{output_suffix}.csv")
    final_df.to_csv(final_path, index=False)

    # Summary
    valid = results_df['predicted_srs'].dropna()
    print(f"\nValid predictions: {len(valid)}/{len(results_df)}")
    if len(valid) > 0:
        print(f"  Mean SRS:   {valid.mean():.4f}")
        print(f"  Median SRS: {valid.median():.4f}")
        print(f"  Std SRS:    {valid.std():.4f}")

    # Cleanup GPU
    del model, tokenizer
    clear_gpu_memory()

    return final_df


# =============================================================================
# NEW: MARKET CATEGORY CLASSIFICATION
# =============================================================================

def classify_market_category(question):
    """Classify a market question into a category using keyword matching.

    Categories: Sports, Politics, Crypto, Economics, Entertainment, Science, Other
    """
    q = str(question).lower()

    sports_kw = ['nba', 'nfl', 'mlb', 'nhl', 'soccer', 'football', 'basketball',
                 'baseball', 'hockey', 'tennis', 'golf', 'boxing', 'ufc', 'mma',
                 'match', 'game', 'vs.', 'vs ', 'league', 'championship', 'world cup',
                 'super bowl', 'playoffs', 'finals', 'win', 'score', 'ncaa',
                 'premier league', 'la liga', 'serie a', 'f1', 'grand prix',
                 'olympics', 'medal', 'athlete']

    politics_kw = ['trump', 'biden', 'election', 'president', 'congress', 'senate',
                   'democrat', 'republican', 'vote', 'poll', 'governor', 'mayor',
                   'legislation', 'law', 'bill', 'supreme court', 'political',
                   'impeach', 'cabinet', 'secretary', 'primary', 'nominee',
                   'campaign', 'party', 'tariff', 'sanctions', 'nato', 'un ',
                   'diplomatic', 'hamas', 'israel', 'ukraine', 'russia', 'china',
                   'military', 'war', 'ceasefire', 'treaty']

    crypto_kw = ['bitcoin', 'btc', 'ethereum', 'eth', 'crypto', 'token', 'blockchain',
                 'defi', 'nft', 'coin', 'solana', 'dogecoin', 'binance', 'coinbase',
                 'mining', 'staking', 'dao', 'web3', 'farcaster', 'polymarket']

    econ_kw = ['gdp', 'inflation', 'interest rate', 'fed ', 'federal reserve',
               'stock', 'market', 's&p', 'nasdaq', 'dow jones', 'earnings',
               'unemployment', 'jobs report', 'cpi', 'recession', 'economy',
               'trade', 'export', 'import', 'oil price', 'commodity']

    entertainment_kw = ['movie', 'film', 'oscar', 'grammy', 'emmy', 'album',
                        'song', 'netflix', 'disney', 'box office', 'celebrity',
                        'actor', 'actress', 'tv show', 'series', 'streaming',
                        'spotify', 'youtube', 'tiktok', 'instagram', 'twitter']

    science_kw = ['ai ', 'artificial intelligence', 'openai', 'gpt', 'nasa',
                  'spacex', 'launch', 'climate', 'temperature', 'weather',
                  'vaccine', 'covid', 'fda', 'drug', 'study', 'research',
                  'science', 'technology', 'robot']

    # Check in order of specificity
    for kw in sports_kw:
        if kw in q: return 'Sports'
    for kw in crypto_kw:
        if kw in q: return 'Crypto'
    for kw in politics_kw:
        if kw in q: return 'Politics'
    for kw in econ_kw:
        if kw in q: return 'Economics'
    for kw in entertainment_kw:
        if kw in q: return 'Entertainment'
    for kw in science_kw:
        if kw in q: return 'Science'

    return 'Other'


# =============================================================================
# STAGE 7: HYPOTHESIS TESTING (REVISED)
# =============================================================================

def stage_7_hypothesis_testing(df=None):
    """Run hypothesis tests with category fixed effects and robustness checks.

    REVISIONS from reviewer response:
    [1a] Category fixed effects
    [1b] R², diagnostics, VIF, Breusch-Pagan
    [1e] Robustness: alt volume thresholds, with/without winsorization
    [6a] Bonferroni correction
    [6b] Interpret insignificant coefficients
    """
    import statsmodels.formula.api as smf
    import statsmodels.stats.api as sms
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from scipy.stats.mstats import winsorize
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import seaborn as sns

    print("\n" + "=" * 70)
    print("STAGE 7: HYPOTHESIS TESTING (REVISED)")
    print("=" * 70)

    results_dir = CONFIG["RESULTS_DIR"]
    os.makedirs(results_dir, exist_ok=True)

    if df is None:
        path = os.path.join(CONFIG["SRS_OUTPUT_DIR"], "analysis_with_srs.csv")
        df = pd.read_csv(path)
        print(f"Loaded {len(df)} markets")

    # ---- Data Preparation ----
    df['created_at'] = pd.to_datetime(df['created_at'], format='mixed', errors='coerce')
    df['resolved_at'] = pd.to_datetime(df['resolved_at'], format='mixed', errors='coerce')
    df['duration_days'] = (df['resolved_at'] - df['created_at']).dt.days

    df = df.dropna(subset=['predicted_srs', 'volume', 'final_price']).copy()
    df = df[df['duration_days'] > 0]

    # Convergence gap
    df['convergence_gap'] = np.minimum(df['final_price'], 1 - df['final_price'])

    # Log transforms
    df['log_volume'] = np.log(df['volume'] + 1)
    df['log_duration'] = np.log(df['duration_days'] + 1)

    # [NEW] Category classification
    df['category'] = df['question'].apply(classify_market_category)
    print(f"\nCategory Distribution:")
    print(df['category'].value_counts().to_string())

    print(f"\nData ready: {len(df)} markets")

    # ---- Helper: Run and report a regression ----
    def run_regression(formula, data, label, dep_var_name):
        print(f"\n{'='*60}")
        print(f"{label}")
        print(f"{'='*60}")
        print(f"Formula: {formula}")
        print(f"N = {len(data)}")

        model = smf.ols(formula, data=data).fit()

        # Full summary table
        print(model.summary().tables[1])
        print(f"\nR²          = {model.rsquared:.4f}")
        print(f"Adjusted R² = {model.rsquared_adj:.4f}")
        print(f"F-statistic = {model.fvalue:.2f} (p = {model.f_pvalue:.6f})")
        print(f"N           = {int(model.nobs)}")

        # [1b] Breusch-Pagan heteroskedasticity test
        try:
            bp_test = sms.het_breuschpagan(model.resid, model.model.exog)
            print(f"\nBreusch-Pagan test:")
            print(f"  LM statistic = {bp_test[0]:.4f}")
            print(f"  p-value      = {bp_test[1]:.4f}")
            if bp_test[1] < 0.05:
                print("  → Heteroskedasticity detected. Consider robust SEs.")
                # Re-estimate with robust SEs
                model_robust = smf.ols(formula, data=data).fit(cov_type='HC3')
                print("\n  Robust Standard Errors (HC3):")
                print(model_robust.summary().tables[1])
        except Exception as e:
            print(f"  Breusch-Pagan test failed: {e}")

        # [1b] VIF (only for numeric predictors)
        try:
            numeric_cols = [c for c in model.model.exog_names if c != 'Intercept'
                           and not c.startswith('C(')]
            if len(numeric_cols) >= 2:
                # Build numeric-only design matrix
                from patsy import dmatrix
                X_numeric = data[['predicted_srs', 'log_duration']].dropna()
                if len(X_numeric) > 0:
                    print(f"\nVIF (numeric predictors):")
                    for i, col in enumerate(X_numeric.columns):
                        vif = variance_inflation_factor(X_numeric.values, i)
                        print(f"  {col}: {vif:.2f}")
        except Exception as e:
            print(f"  VIF calculation failed: {e}")

        return model

    # ================================================================
    # MAIN REGRESSIONS (with category fixed effects)
    # ================================================================

    # Winsorize for main models
    df_w = df.copy()
    df_w['predicted_srs'] = winsorize(df_w['predicted_srs'], limits=[0.01, 0.01])
    df_w['log_volume'] = winsorize(df_w['log_volume'], limits=[0.01, 0.01])
    df_w['log_duration'] = winsorize(df_w['log_duration'], limits=[0.01, 0.01])
    df_w['convergence_gap'] = winsorize(df_w['convergence_gap'], limits=[0.01, 0.01])

    # --- H1: Liquidity (baseline — matches original paper) ---
    liq_baseline = run_regression(
        "log_volume ~ predicted_srs + log_duration",
        df_w, "H1 LIQUIDITY — BASELINE (no fixed effects)", "log(Volume)"
    )

    # --- H1: Liquidity (with category fixed effects) [1a] ---
    liq_fe = run_regression(
        "log_volume ~ predicted_srs + log_duration + C(category)",
        df_w, "H1 LIQUIDITY — WITH CATEGORY FIXED EFFECTS", "log(Volume)"
    )

    # --- H2: Convergence (baseline) ---
    conv_baseline = run_regression(
        "convergence_gap ~ predicted_srs + log_volume",
        df_w, "H2 CONVERGENCE — BASELINE (no fixed effects)", "Convergence Gap"
    )

    # --- H2: Convergence (with category fixed effects) [1a] ---
    conv_fe = run_regression(
        "convergence_gap ~ predicted_srs + log_volume + C(category)",
        df_w, "H2 CONVERGENCE — WITH CATEGORY FIXED EFFECTS", "Convergence Gap"
    )

    # ================================================================
    # [6a] BONFERRONI CORRECTION
    # ================================================================
    print(f"\n{'='*60}")
    print("MULTIPLE TESTING CORRECTION")
    print(f"{'='*60}")
    p_h1 = liq_fe.pvalues.get('predicted_srs', 1.0)
    p_h2 = conv_fe.pvalues.get('predicted_srs', 1.0)
    bonferroni_alpha = 0.05 / 2
    print(f"Number of tests: 2")
    print(f"Bonferroni-adjusted α = {bonferroni_alpha}")
    print(f"H1 SRS p-value: {p_h1:.6f} → {'SIGNIFICANT' if p_h1 < bonferroni_alpha else 'NOT SIGNIFICANT'}")
    print(f"H2 SRS p-value: {p_h2:.6f} → {'SIGNIFICANT' if p_h2 < bonferroni_alpha else 'NOT SIGNIFICANT'}")

    # ================================================================
    # [1e] ROBUSTNESS CHECKS
    # ================================================================
    print(f"\n{'='*60}")
    print("ROBUSTNESS CHECKS")
    print(f"{'='*60}")

    # --- Alternative volume thresholds ---
    for threshold in [1000, 10000]:
        df_alt = df[df['volume'] >= threshold].copy()
        df_alt_w = df_alt.copy()
        df_alt_w['predicted_srs'] = winsorize(df_alt_w['predicted_srs'], limits=[0.01, 0.01])
        df_alt_w['log_volume'] = winsorize(df_alt_w['log_volume'], limits=[0.01, 0.01])
        df_alt_w['log_duration'] = winsorize(df_alt_w['log_duration'], limits=[0.01, 0.01])

        print(f"\n--- Volume threshold: ${threshold:,} (N={len(df_alt_w)}) ---")
        rob_model = smf.ols(
            "log_volume ~ predicted_srs + log_duration + C(category)",
            data=df_alt_w
        ).fit()
        coef = rob_model.params.get('predicted_srs', np.nan)
        pval = rob_model.pvalues.get('predicted_srs', np.nan)
        print(f"  SRS coefficient: {coef:.3f} (p = {pval:.4f})")
        print(f"  R² = {rob_model.rsquared:.4f}")

    # --- Without winsorization ---
    print(f"\n--- Without winsorization (N={len(df)}) ---")
    rob_no_wins = smf.ols(
        "log_volume ~ predicted_srs + log_duration + C(category)",
        data=df
    ).fit()
    coef = rob_no_wins.params.get('predicted_srs', np.nan)
    pval = rob_no_wins.pvalues.get('predicted_srs', np.nan)
    print(f"  SRS coefficient: {coef:.3f} (p = {pval:.4f})")
    print(f"  R² = {rob_no_wins.rsquared:.4f}")

    # --- Quantile regression ---
    print(f"\n--- Quantile Regression (median) ---")
    try:
        qr_model = smf.quantreg(
            "log_volume ~ predicted_srs + log_duration + C(category)",
            data=df_w
        ).fit(q=0.5)
        coef = qr_model.params.get('predicted_srs', np.nan)
        pval = qr_model.pvalues.get('predicted_srs', np.nan)
        print(f"  SRS coefficient (median): {coef:.3f} (p = {pval:.4f})")
    except Exception as e:
        print(f"  Quantile regression failed: {e}")

    # ================================================================
    # PLOTS
    # ================================================================
    print(f"\n--- Generating Plots ---")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. SRS vs Volume scatter
    ax1 = axes[0, 0]
    for cat in df_w['category'].unique():
        subset = df_w[df_w['category'] == cat]
        ax1.scatter(subset['predicted_srs'], subset['log_volume'],
                    alpha=0.5, s=30, label=cat)
    ax1.set_xlabel('Semantic Risk Score (SRS)')
    ax1.set_ylabel('log(Volume)')
    ax1.set_title('SRS vs Trading Volume by Category')
    ax1.legend(fontsize=8)

    # 2. SRS vs Convergence Gap
    ax2 = axes[0, 1]
    sns.regplot(x='predicted_srs', y='convergence_gap', data=df_w, ax=ax2,
                scatter_kws={'alpha': 0.4}, line_kws={'color': 'red'})
    ax2.set_xlabel('Semantic Risk Score (SRS)')
    ax2.set_ylabel('Convergence Gap')
    ax2.set_title('SRS vs Price Convergence')

    # 3. Category volume distribution
    ax3 = axes[1, 0]
    category_stats = df_w.groupby('category').agg(
        median_vol=('volume', 'median'),
        mean_srs=('predicted_srs', 'mean'),
        count=('volume', 'count')
    ).sort_values('median_vol', ascending=True)
    ax3.barh(category_stats.index, category_stats['median_vol'] / 1000)
    ax3.set_xlabel('Median Volume ($K)')
    ax3.set_title('Median Volume by Market Category')

    # 4. Residuals plot for liquidity model
    ax4 = axes[1, 1]
    ax4.scatter(liq_fe.fittedvalues, liq_fe.resid, alpha=0.4, s=20)
    ax4.axhline(y=0, color='red', linestyle='--')
    ax4.set_xlabel('Fitted Values')
    ax4.set_ylabel('Residuals')
    ax4.set_title('Liquidity Model: Residuals vs Fitted')

    plt.tight_layout()
    plot_path = os.path.join(results_dir, 'hypothesis_tests.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Plots saved to {plot_path}")

    # ================================================================
    # FORMATTED RESULTS FOR PAPER
    # ================================================================
    print(f"\n{'='*60}")
    print("FORMATTED RESULTS FOR PAPER")
    print(f"{'='*60}")

    for label, model in [("Table 2: Liquidity (with FE)", liq_fe),
                          ("Table 3: Convergence (with FE)", conv_fe)]:
        print(f"\n{label}")
        print("-" * 70)
        print(f"{'Variable':<25} {'Coef':>10} {'Std.Err':>10} {'t':>10} {'p-value':>10}")
        print("-" * 70)
        for var in model.params.index:
            display_var = var[:25]
            print(f"{display_var:<25} {model.params[var]:>10.3f} {model.bse[var]:>10.3f} "
                  f"{model.tvalues[var]:>10.2f} {model.pvalues[var]:>10.4f}")
        print(f"\nN = {int(model.nobs)}, R² = {model.rsquared:.4f}, "
              f"Adj R² = {model.rsquared_adj:.4f}")

    # [6b] Note on insignificant coefficient
    conv_vol_p = conv_fe.pvalues.get('log_volume', 1.0)
    if conv_vol_p > 0.05:
        print(f"\nNote: log(Volume) is insignificant in the convergence model "
              f"(p = {conv_vol_p:.3f}), indicating the SRS-convergence effect "
              f"operates independently of liquidity levels.")

    return {
        'df': df_w,
        'liq_baseline': liq_baseline,
        'liq_fe': liq_fe,
        'conv_baseline': conv_baseline,
        'conv_fe': conv_fe,
    }


# =============================================================================
# STAGE 8: PORTFOLIO OPTIMIZATION (REFRAMED)
# =============================================================================

def stage_8_portfolio(df=None):
    """Portfolio analysis reframed as capital efficiency screening.

    REVISION: Reframed per reviewer — this is a capital efficiency
    demonstration, not a performance claim.
    """
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    print("\n" + "=" * 70)
    print("STAGE 8: CAPITAL EFFICIENCY ANALYSIS (SRS-Based Screening)")
    print("=" * 70)

    results_dir = CONFIG["RESULTS_DIR"]
    os.makedirs(results_dir, exist_ok=True)

    if df is None:
        path = os.path.join(CONFIG["SRS_OUTPUT_DIR"], "analysis_with_srs.csv")
        df = pd.read_csv(path)

    df['created_at'] = pd.to_datetime(df['created_at'], format='mixed', errors='coerce')
    df['resolved_at'] = pd.to_datetime(df['resolved_at'], format='mixed', errors='coerce')
    df['duration_days'] = (df['resolved_at'] - df['created_at']).dt.days
    df = df.dropna(subset=['predicted_srs', 'volume']).copy()

    threshold = CONFIG["SRS_THRESHOLD"]
    gamma = CONFIG["GAMMA"]

    # Naive portfolio
    naive_df = df.copy()
    naive_df['weight'] = 1 / len(naive_df)

    # Clarity portfolio
    clarity_df = df.copy()
    clarity_df['weight'] = 0.0
    mask = clarity_df['predicted_srs'] <= threshold
    eligible = clarity_df[mask].copy()

    if len(eligible) > 0:
        scores = (1 - eligible['predicted_srs']) ** gamma
        weights = scores / scores.sum()
        clarity_df.loc[mask, 'weight'] = weights.values

    naive_active = naive_df[naive_df['weight'] > 0]
    clarity_active = clarity_df[clarity_df['weight'] > 0]

    # Metrics
    def calc_metrics(active, name):
        dead = active[active['volume'] < CONFIG['MIN_VIABLE_VOLUME']]
        return {
            'Portfolio': name,
            'Markets Included': len(active),
            'Mean SRS': round(active['predicted_srs'].mean(), 4),
            'Median SRS': round(active['predicted_srs'].median(), 4),
            'Weighted SRS': round((active['weight'] * active['predicted_srs']).sum(), 4),
            'Median Volume': active['volume'].median(),
            'Mean Volume': active['volume'].mean(),
            'Dead Markets': len(dead),
            'Dead Market Rate (%)': round(100 * len(dead) / len(active), 1),
        }

    naive_m = calc_metrics(naive_active, "Naive")
    clarity_m = calc_metrics(clarity_active, f"Clarity (τ={threshold})")

    # Print comparison
    print(f"\n{'Metric':<30} {'Naive':>15} {'Clarity':>15} {'Change':>12}")
    print("-" * 72)
    for key in ['Markets Included', 'Mean SRS', 'Weighted SRS', 'Median Volume', 'Mean Volume']:
        n = naive_m[key]
        c = clarity_m[key]
        if n != 0:
            pct = 100 * (c / n - 1)
            if isinstance(n, float) and n < 1:
                print(f"{key:<30} {n:>15.3f} {c:>15.3f} {pct:>+11.0f}%")
            else:
                print(f"{key:<30} {str(n):>15} {str(c):>15} {pct:>+11.0f}%")
        else:
            print(f"{key:<30} {str(n):>15} {str(c):>15} {'N/A':>12}")

    # Sensitivity analysis
    print(f"\n{'='*60}")
    print("SENSITIVITY ANALYSIS")
    print(f"{'='*60}")
    thresholds = [0.03, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25]
    sens_results = []
    for tau in thresholds:
        port = df[df['predicted_srs'] <= tau]
        if len(port) == 0:
            continue
        sens_results.append({
            'Threshold': tau,
            'Markets': len(port),
            'Filter Rate (%)': round(100 * (len(df) - len(port)) / len(df), 1),
            'Mean SRS': round(port['predicted_srs'].mean(), 3),
            'Median Volume ($)': round(port['volume'].median(), 0),
        })
    sens_df = pd.DataFrame(sens_results)
    print(sens_df.to_string(index=False))

    # Table for paper (reframed)
    print(f"\n{'='*60}")
    print("TABLE FOR PAPER: Capital Efficiency — SRS-Based Screening")
    print(f"{'='*60}")
    print("| Metric | All Markets | SRS-Filtered (τ≤0.10) | Change |")
    print("|--------|-------------|----------------------|--------|")
    print(f"| Markets | {naive_m['Markets Included']} | {clarity_m['Markets Included']} | "
          f"{100*(clarity_m['Markets Included']/naive_m['Markets Included'] - 1):+.0f}% |")
    print(f"| Mean SRS | {naive_m['Mean SRS']:.3f} | {clarity_m['Mean SRS']:.3f} | "
          f"{100*(clarity_m['Mean SRS']/naive_m['Mean SRS'] - 1):+.0f}% |")
    print(f"| Median Volume | ${naive_m['Median Volume']:,.0f} | ${clarity_m['Median Volume']:,.0f} | "
          f"{100*(clarity_m['Median Volume']/naive_m['Median Volume'] - 1):+.0f}% |")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax1 = axes[0]
    ax1.hist(naive_active['predicted_srs'], bins=15, alpha=0.5,
             label='All Markets', color='steelblue', density=True)
    ax1.hist(clarity_active['predicted_srs'], bins=15, alpha=0.5,
             label='SRS-Filtered', color='forestgreen', density=True)
    ax1.axvline(x=threshold, color='red', linestyle='--', linewidth=2, label=f'τ = {threshold}')
    ax1.set_xlabel('Semantic Risk Score')
    ax1.set_ylabel('Density')
    ax1.set_title('SRS Distribution: All vs Filtered')
    ax1.legend()

    ax2 = axes[1]
    ax2.plot(sens_df['Threshold'], sens_df['Median Volume ($)'] / 1000,
             marker='o', linewidth=2, markersize=8, color='forestgreen')
    ax2.axvline(x=0.10, color='red', linestyle='--', alpha=0.7, label='τ = 0.10')
    ax2.set_xlabel('SRS Threshold (τ)')
    ax2.set_ylabel('Median Volume ($K)')
    ax2.set_title('Liquidity by SRS Threshold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(results_dir, 'capital_efficiency.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\nPlot saved to {plot_path}")

    return sens_df


# =============================================================================
# STAGE 9: SECOND MODEL ROBUSTNESS [2c]
# =============================================================================

def stage_9_second_model(pm_df=None):
    """Run SRS with Llama-3-8B for inter-rater reliability."""
    print("\n" + "=" * 70)
    print("STAGE 9: SECOND MODEL ROBUSTNESS (Llama-3-8B)")
    print("=" * 70)

    # Run predictions with secondary model
    final_df = stage_6_predict_polymarket(
        pm_df=pm_df,
        model_name=CONFIG["SECONDARY_MODEL"],
        temperature=CONFIG["TEMPERATURE_PRIMARY"],
        output_suffix="_llama"
    )

    # Load primary model predictions
    primary_path = os.path.join(CONFIG["SRS_OUTPUT_DIR"], "analysis_with_srs.csv")
    primary_df = pd.read_csv(primary_path)

    # Compare
    merged = primary_df[['condition_id', 'predicted_srs']].merge(
        final_df[['condition_id', 'predicted_srs_llama']],
        on='condition_id', how='inner'
    )
    merged = merged.dropna(subset=['predicted_srs', 'predicted_srs_llama'])

    if len(merged) >= 3:
        from scipy.stats import spearmanr, pearsonr

        sp, sp_p = spearmanr(merged['predicted_srs'], merged['predicted_srs_llama'])
        pe, pe_p = pearsonr(merged['predicted_srs'], merged['predicted_srs_llama'])

        print(f"\n{'='*60}")
        print("INTER-MODEL RELIABILITY")
        print(f"{'='*60}")
        print(f"N = {len(merged)}")
        print(f"Spearman ρ = {sp:.4f} (p = {sp_p:.6f})")
        print(f"Pearson r  = {pe:.4f} (p = {pe_p:.6f})")
        print(f"\nInterpretation: {'Strong' if sp > 0.7 else 'Moderate' if sp > 0.4 else 'Weak'} "
              f"inter-model agreement.")

        # Run regressions with secondary model SRS
        print(f"\n--- Regressions with Llama SRS ---")
        import statsmodels.formula.api as smf
        from scipy.stats.mstats import winsorize

        reg_df = final_df.dropna(subset=['predicted_srs_llama', 'volume']).copy()
        reg_df['created_at'] = pd.to_datetime(reg_df['created_at'], format='mixed', errors='coerce')
        reg_df['resolved_at'] = pd.to_datetime(reg_df['resolved_at'], format='mixed', errors='coerce')
        reg_df['duration_days'] = (reg_df['resolved_at'] - reg_df['created_at']).dt.days
        reg_df = reg_df[reg_df['duration_days'] > 0].copy()
        reg_df['log_volume'] = np.log(reg_df['volume'] + 1)
        reg_df['log_duration'] = np.log(reg_df['duration_days'] + 1)
        reg_df['category'] = reg_df['question'].apply(classify_market_category)

        if len(reg_df) > 10:
            model = smf.ols(
                "log_volume ~ predicted_srs_llama + log_duration + C(category)",
                data=reg_df
            ).fit()
            coef = model.params.get('predicted_srs_llama', np.nan)
            pval = model.pvalues.get('predicted_srs_llama', np.nan)
            print(f"  Llama SRS → Volume: coef = {coef:.3f}, p = {pval:.4f}, R² = {model.rsquared:.4f}")

    return merged


# =============================================================================
# STAGE 10: TEMPERATURE SENSITIVITY [4c]
# =============================================================================

def stage_10_temperature_sensitivity(pm_df=None):
    """Compare SRS at temp=0.0 vs temp=0.6."""
    print("\n" + "=" * 70)
    print("STAGE 10: TEMPERATURE SENSITIVITY CHECK")
    print("=" * 70)

    # Run predictions at elevated temperature
    final_df = stage_6_predict_polymarket(
        pm_df=pm_df,
        model_name=CONFIG["PRIMARY_MODEL"],
        temperature=CONFIG["TEMPERATURE_SENSITIVITY"],
        output_suffix="_temp06"
    )

    # Load temp=0 predictions
    primary_path = os.path.join(CONFIG["SRS_OUTPUT_DIR"], "analysis_with_srs.csv")
    primary_df = pd.read_csv(primary_path)

    # Compare
    merged = primary_df[['condition_id', 'predicted_srs']].merge(
        final_df[['condition_id', 'predicted_srs_temp06']],
        on='condition_id', how='inner'
    )
    merged = merged.dropna(subset=['predicted_srs', 'predicted_srs_temp06'])

    if len(merged) >= 3:
        from scipy.stats import spearmanr

        sp, sp_p = spearmanr(merged['predicted_srs'], merged['predicted_srs_temp06'])

        print(f"\n{'='*60}")
        print("TEMPERATURE SENSITIVITY RESULTS")
        print(f"{'='*60}")
        print(f"N = {len(merged)}")
        print(f"Spearman ρ (temp=0.0 vs temp=0.6) = {sp:.4f} (p = {sp_p:.6f})")
        print(f"Mean diff = {(merged['predicted_srs'] - merged['predicted_srs_temp06']).mean():.4f}")
        print(f"Max diff  = {(merged['predicted_srs'] - merged['predicted_srs_temp06']).abs().max():.4f}")

    return merged


# =============================================================================
# STAGE 11: CLASSIFICATION PERFORMANCE [2b]
# =============================================================================

def stage_11_classification_performance(uma_validation_df=None):
    """Evaluate SRS as a binary classifier for disputed vs undisputed."""
    print("\n" + "=" * 70)
    print("STAGE 11: CLASSIFICATION PERFORMANCE (AUC-ROC)")
    print("=" * 70)

    if uma_validation_df is None:
        path = os.path.join(CONFIG["UMA_OUTPUT_DIR"], "validation_results.csv")
        uma_validation_df = pd.read_csv(path)

    df = uma_validation_df.dropna(subset=['predicted_srs', 'uma_bernoulli_variance']).copy()

    # Binary label: disputed = variance > 0.05
    df['disputed'] = (df['uma_bernoulli_variance'] > 0.05).astype(int)

    print(f"N = {len(df)}")
    print(f"Disputed (variance > 0.05): {df['disputed'].sum()} ({100*df['disputed'].mean():.1f}%)")
    print(f"Undisputed: {(1-df['disputed']).sum()} ({100*(1-df['disputed'].mean()):.1f}%)")

    if df['disputed'].sum() < 5 or (1-df['disputed']).sum() < 5:
        print("Insufficient class balance for classification analysis.")
        return None

    from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, roc_curve

    y_true = df['disputed']
    y_scores = df['predicted_srs']

    # AUC-ROC
    auc = roc_auc_score(y_true, y_scores)
    print(f"\nAUC-ROC = {auc:.4f}")

    # Find optimal threshold (Youden's J)
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    best_threshold = thresholds[best_idx]

    # Classification at optimal threshold
    y_pred = (y_scores >= best_threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0
    )

    print(f"\nOptimal threshold (Youden's J): {best_threshold:.4f}")
    print(f"Precision = {precision:.4f}")
    print(f"Recall    = {recall:.4f}")
    print(f"F1 Score  = {f1:.4f}")

    # Also at fixed thresholds
    print(f"\nPerformance at fixed thresholds:")
    for t in [0.05, 0.08, 0.10, 0.15]:
        y_p = (y_scores >= t).astype(int)
        p, r, f, _ = precision_recall_fscore_support(y_true, y_p, average='binary', zero_division=0)
        print(f"  τ={t:.2f}: Precision={p:.3f}, Recall={r:.3f}, F1={f:.3f}")

    # ROC curve plot
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    results_dir = CONFIG["RESULTS_DIR"]
    os.makedirs(results_dir, exist_ok=True)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'SRS (AUC = {auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
    ax.plot(fpr[best_idx], tpr[best_idx], 'ro', markersize=10,
            label=f'Optimal (τ = {best_threshold:.3f})')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve: SRS as Dispute Classifier')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plot_path = os.path.join(results_dir, 'roc_curve.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\nROC curve saved to {plot_path}")

    return {'auc': auc, 'precision': precision, 'recall': recall, 'f1': f1,
            'optimal_threshold': best_threshold}


# =============================================================================
# MAIN: ORCHESTRATOR
# =============================================================================

if __name__ == "__main__":
    """
    Run stages sequentially. Comment out stages you've already completed.

    GPU stages (3, 6, 9, 10) need a GPU runtime.
    Data collection stages (1, 4) need internet access.
    Analysis stages (7, 8, 11) are CPU-only.
    """

    print("=" * 70)
    print("SEMANTIC RISK IN PREDICTION MARKETS — UNIFIED EXPERIMENT")
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)

    # ---- PHASE A: DATA COLLECTION (run once, then comment out) ----

    # Stage 1: UMA data collection
    # uma_df = stage_1_collect_uma()

    # Stage 2: UMA cleanup and few-shot split
    # few_shot_df, analysis_df = stage_2_uma_cleanup()

    # Stage 4: Polymarket data collection
    # pm_df = stage_4_collect_polymarket()

    # Stage 5: Polymarket cleanup
    # pm_df = stage_5_clean_polymarket()

    # ---- PHASE B: SRS PREDICTION (requires GPU) ----

    # Stage 3: Validate SRS on UMA
    # uma_validated = stage_3_validate_uma()

    # Stage 6: SRS prediction on Polymarket (primary model)
    # pm_with_srs = stage_6_predict_polymarket()

    # ---- PHASE C: ANALYSIS (CPU-only) ----

    # Stage 7: Hypothesis testing with fixed effects & robustness
    # results = stage_7_hypothesis_testing()

    # Stage 8: Portfolio / capital efficiency analysis
    # sens_df = stage_8_portfolio()

    # ---- PHASE D: ROBUSTNESS (requires GPU) ----

    # Stage 9: Second model (Llama-3-8B) for inter-rater reliability
    # model_comparison = stage_9_second_model()

    # Stage 10: Temperature sensitivity
    # temp_comparison = stage_10_temperature_sensitivity()

    # ---- PHASE E: ADDITIONAL VALIDATION (CPU-only) ----

    # Stage 11: Classification performance (AUC-ROC)
    # class_results = stage_11_classification_performance()

    print("\n" + "=" * 70)
    print("EXPERIMENT COMPLETE")
    print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)

SEMANTIC RISK IN PREDICTION MARKETS — UNIFIED EXPERIMENT
Started: 2026-04-12 17:30:46

EXPERIMENT COMPLETE
Finished: 2026-04-12 17:30:46


**Stage 1**

In [ ]:
uma_df = stage_1_collect_uma()


STAGE 1: UMA DATA COLLECTION
UMA connection successful.
  Offset 0: Got 100. Total: 100
  Offset 100: Got 100. Total: 200
  Offset 200: Got 100. Total: 300
  Offset 300: Got 100. Total: 400
  Offset 400: Got 100. Total: 500
  Offset 500: Got 100. Total: 600
  Offset 600: Got 100. Total: 700
  Offset 700: Got 100. Total: 800
  Offset 800: Got 100. Total: 900
  Offset 900: Got 100. Total: 1000
  Offset 1000: Got 100. Total: 1100
  Offset 1100: Got 100. Total: 1200
  Offset 1200: Got 100. Total: 1300
  Offset 1300: Got 100. Total: 1400
  Offset 1400: Got 100. Total: 1500
  Offset 1500: Got 100. Total: 1600
  Offset 1600: Got 100. Total: 1700
  Offset 1700: Got 100. Total: 1800
  Offset 1800: Got 100. Total: 1900
  Offset 1900: Got 100. Total: 2000
  Offset 2000: Got 100. Total: 2100
  Offset 2100: Got 100. Total: 2200
  Offset 2200: Got 100. Total: 2300
  Offset 2300: Got 100. Total: 2400
  Offset 2400: Got 100. Total: 2500
  Offset 2500: Got 100. Total: 2600
  Offset 2600: Got 100. Tota

**Stage 2**

In [ ]:
few_shot_df, analysis_df = stage_2_uma_cleanup()


STAGE 2: UMA CLEANUP & FEW-SHOT SPLIT
Loaded 817 markets from ./prediction_market_data/01_uma_all_markets.csv
Few-Shot Set: 13 markets
Analysis Set: 804 markets
All 10 few-shot examples identified.
Saved to ./prediction_market_data/


**Stage 3A**

In [ ]:
!pip install -U bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.5 MB/s eta 0:00:00


**Stage 3B**

In [ ]:
uma_validated = stage_3_validate_uma()


STAGE 3: SRS VALIDATION ON UMA (Primary Model)
Loaded 804 markets from ./prediction_market_data/03_analysis_set.csv
GPU memory: 0.00 GB
Loading model: Qwen/Qwen3-8B


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  Checkpoint: 750 predictions


Validating SRS: 100%|█████████▉| 800/804 [1:47:07<00:36,  9.11s/it]

  Checkpoint: 800 predictions


Validating SRS: 100%|██████████| 804/804 [1:47:33<00:00,  8.03s/it]



VALIDATION RESULTS
N = 804
Spearman ρ = 0.1429 (p = 0.000048)
Pearson r  = 0.0734 (p = 0.037464)
MAE        = 0.0424
Saved to ./prediction_market_data/validation_results.csv
GPU memory: 0.01 GB


**Stage 4**

In [ ]:
pm_df = stage_4_collect_polymarket()


STAGE 4: POLYMARKET DATA COLLECTION
  Offset 8000: Got 100, kept 100. Total: 100
  Offset 8100: Got 100, kept 100. Total: 200
  Offset 8200: Got 100, kept 100. Total: 300
  Offset 8300: Got 100, kept 100. Total: 400
  Offset 8400: Got 100, kept 100. Total: 500
  Offset 8500: Got 100, kept 100. Total: 600
  Offset 8600: Got 100, kept 100. Total: 700
  Offset 8700: Got 100, kept 100. Total: 800
  Offset 8800: Got 100, kept 100. Total: 900
  Offset 8900: Got 100, kept 100. Total: 1000
Valid markets: 617
After volume filter (>$5000): 497

Fetching price histories...


100%|██████████| 497/497 [11:28<00:00,  1.39s/it]


Saved 497 markets to ./polymarket_data/05_analysis_ready.csv


**Stage 5**

In [ ]:
pm_df = stage_5_clean_polymarket()


STAGE 5: POLYMARKET CLEANUP
Original count: 497
Ready for SRS prediction: 246 markets


**Stage 6**

In [ ]:
pm_with_srs = stage_6_predict_polymarket()


STAGE 6: SRS PREDICTION ON POLYMARKET (Qwen3-8B, temp=0.0)
Loaded 246 markets
GPU memory: 0.01 GB
Loading model: Qwen/Qwen3-8B


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Model loaded. GPU memory: 5.67 GB


Predicting SRS: 100%|██████████| 246/246 [29:35<00:00,  7.22s/it]



Valid predictions: 246/246
  Mean SRS:   0.0436
  Median SRS: 0.0300
  Std SRS:    0.0323
GPU memory: 0.01 GB


**Stage 7**

In [ ]:
results = stage_7_hypothesis_testing()


STAGE 7: HYPOTHESIS TESTING (REVISED)
Loaded 246 markets

Category Distribution:
category
Other            74
Sports           73
Politics         49
Crypto           28
Science          17
Economics         2
Entertainment     1

Data ready: 244 markets

H1 LIQUIDITY — BASELINE (no fixed effects)
Formula: log_volume ~ predicted_srs + log_duration
N = 244
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         8.7812      0.368     23.841      0.000       8.056       9.507
predicted_srs   -10.4753      4.589     -2.283      0.023     -19.515      -1.436
log_duration      1.0292      0.088     11.651      0.000       0.855       1.203

R²          = 0.3607
Adjusted R² = 0.3554
F-statistic = 67.98 (p = 0.000000)
N           = 244

Breusch-Pagan test:
  LM statistic = 39.6611
  p-value      = 0.0000
  → Heteroskedasticity detected. Consider robust SEs.

  Robust S

**Stage 8**

In [ ]:
sens_df = stage_8_portfolio()


STAGE 8: CAPITAL EFFICIENCY ANALYSIS (SRS-Based Screening)

Metric                                   Naive         Clarity       Change
------------------------------------------------------------------------
Markets Included                           246             237          -4%
Mean SRS                                 0.044           0.039         -10%
Weighted SRS                             0.044           0.039         -11%
Median Volume                  59567.160495000004    61818.628205          +4%
Mean Volume                    17819507.09905474 18001845.127361406          +1%

SENSITIVITY ANALYSIS
 Threshold  Markets  Filter Rate (%)  Mean SRS  Median Volume ($)
      0.03      144             41.5     0.023            56261.0
      0.05      192             22.0     0.030            69742.0
      0.08      236              4.1     0.039            62157.0
      0.10      237              3.7     0.039            61819.0
      0.12      237              3.7     0.039    

**Stage 9A**

In [ ]:
CONFIG["SECONDARY_MODEL"] = "Qwen/Qwen2.5-7B-Instruct"

**Stage 9B**

In [ ]:
model_comparison = stage_9_second_model()


STAGE 9: SECOND MODEL ROBUSTNESS (Llama-3-8B)

STAGE 6: SRS PREDICTION ON POLYMARKET (Qwen2.5-7B-Instruct, temp=0.0)
Loaded 246 markets
GPU memory: 0.01 GB
Loading model: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded. GPU memory: 5.21 GB


Predicting SRS: 100%|██████████| 246/246 [18:36<00:00,  4.54s/it]



Valid predictions: 246/246
  Mean SRS:   0.0162
  Median SRS: 0.0000
  Std SRS:    0.0259
GPU memory: 0.01 GB

INTER-MODEL RELIABILITY
N = 246
Spearman ρ = 0.3449 (p = 0.000000)
Pearson r  = 0.4271 (p = 0.000000)

Interpretation: Weak inter-model agreement.

--- Regressions with Llama SRS ---
  Llama SRS → Volume: coef = 1.155, p = 0.8142, R² = 0.5426


**Stage 10**

In [ ]:
temp_comparison = stage_10_temperature_sensitivity()


STAGE 10: TEMPERATURE SENSITIVITY CHECK

STAGE 6: SRS PREDICTION ON POLYMARKET (Qwen3-8B, temp=0.6)
Loaded 246 markets
GPU memory: 0.01 GB
Loading model: Qwen/Qwen3-8B


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Model loaded. GPU memory: 5.67 GB


Predicting SRS: 100%|██████████| 246/246 [29:38<00:00,  7.23s/it]



Valid predictions: 246/246
  Mean SRS:   0.0444
  Median SRS: 0.0300
  Std SRS:    0.0333
GPU memory: 0.01 GB

TEMPERATURE SENSITIVITY RESULTS
N = 246
Spearman ρ (temp=0.0 vs temp=0.6) = 0.7829 (p = 0.000000)
Mean diff = -0.0009
Max diff  = 0.0700


**Stage 11**

In [ ]:
class_results = stage_11_classification_performance()


STAGE 11: CLASSIFICATION PERFORMANCE (AUC-ROC)
N = 804
Disputed (variance > 0.05): 129 (16.0%)
Undisputed: 675 (84.0%)

AUC-ROC = 0.6143

Optimal threshold (Youden's J): 0.0300
Precision = 0.2250
Recall    = 0.7674
F1 Score  = 0.3480

Performance at fixed thresholds:
  τ=0.05: Precision=0.176, Recall=0.295, F1=0.220
  τ=0.08: Precision=0.160, Recall=0.233, F1=0.190
  τ=0.10: Precision=0.303, Recall=0.078, F1=0.123
  τ=0.15: Precision=0.286, Recall=0.062, F1=0.102

ROC curve saved to ./results/roc_curve.png


In [19]:
!zip -r results.zip results/

  adding: results/ (stored 0%)
  adding: results/hypothesis_tests.png (deflated 11%)
  adding: results/roc_curve.png (deflated 9%)
  adding: results/capital_efficiency.png (deflated 13%)
